# XEdu-python 快速上手

安装 XEdu 后，按顺序运行前四个代码单元，你会看到图片中的人体检测结果。之后只需修改 `MY_IMAGE`，就能换成自己的照片。

## 第一步：安装

运行下一格即可安装。它会依次尝试项目内的 wheel、本地 2.1 源码和 GitHub 2.1 分支。`[all]` 会同时安装 OCR、音频和 Gradio 依赖。

安装完成后重启内核，再从后面的单元开始运行。

In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_21 = 'XEdu-python[all] @ git+https://github.com/XEduPro/XEdu-python.git@2.1'

def find_install_target(start=Path.cwd()):
    for folder in (start.resolve(), *start.resolve().parents):
        wheels = sorted((folder / 'wheel').glob('xedu_python-*.whl'), reverse=True)
        if wheels:
            wheel_path = wheels[0]
            return f'{wheel_path}[all]'
        if (folder / 'pyproject.toml').exists() and (folder / 'XEdu' / 'examples').exists():
            return f'{folder}[all]'
    return GITHUB_21

package = find_install_target()

print('正在安装：', package)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    package,
])
print('安装完成。请重启 Notebook 内核，再运行后面的单元。')

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
from IPython.display import display
from PIL import Image

from XEdu.hub import Workflow as wf
from XEdu.hub import workflow as workflow_module
from XEdu.utils import get_similarity

FONT_PATH = Path(workflow_module.__file__).parent / 'font' / 'FZYTK.TTF'
if FONT_PATH.exists():
    font_manager.fontManager.addfont(str(FONT_PATH))
    plt.rcParams['font.sans-serif'] = [font_manager.FontProperties(fname=str(FONT_PATH)).get_name()]
    plt.rcParams['axes.unicode_minus'] = False

print('Python:', sys.executable)
print('可用功能数:', len(wf.support_task()))

## 第二步：选择素材

先直接运行下一格。想换成自己的照片时，修改 `MY_IMAGE`；音频示例会自动生成一个 WAV，也可以改成自己的 `MY_AUDIO`。

In [ ]:
from pathlib import Path
import numpy as np
import soundfile as sf
import XEdu.examples

ASSET_DIR = Path(XEdu.examples.__file__).resolve().parent / 'assets'

# 只需要改这一行，就能使用自己的图片。
MY_IMAGE = ASSET_DIR / 'xedu-vision-scene.png'
MY_ROAD_IMAGE = ASSET_DIR / 'xedu-road-scene.png'
MY_OCR_IMAGE = ASSET_DIR / 'xedu-ocr-poster.png'
MY_TEXT = '这是一节关于人工智能和图像识别的课程。'
OUTPUT_DIR = Path.cwd() / 'xedu_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
MY_AUDIO = OUTPUT_DIR / 'xedu-demo-tone.wav'
if not MY_AUDIO.exists():
    sample_rate = 44100
    time_axis = np.arange(sample_rate, dtype=np.float32) / sample_rate
    sf.write(MY_AUDIO, 0.2 * np.sin(2 * np.pi * 440 * time_axis), sample_rate)

def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{label}不存在：{path}。请在上一个单元替换为自己的文件。')
    return path

def show_bgr(image, title):
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

display(Image.open(require_file(MY_IMAGE, '图片素材')))

## 第三步：先试一次

运行下一格，看看 XEdu 能否找到图片中的人。

<details>
<summary>全部功能（需要时展开）</summary>

- 检测：人体、物体、人脸、手部。
- 关键点：人体 17/26 点、全身 133 点、手部、人脸。
- 图片：分类、OCR、分割、深度、道路感知、风格迁移、着色。
- 比较：图片、文本、音频 embedding，原型分类与图文匹配。
- 文本：问答。
- 扩展：自己的 ONNX/PKL 模型、模型仓库和 LLM。

每项功能都在后面的独立代码单元中。
</details>

In [ ]:
# 找到图片中的人。
detector = wf(task='det_body')
person_boxes, person_image = detector.inference(str(require_file(MY_IMAGE, '图片')), img_type='cv2', thr=0.25)
print(detector.format_output(lang='zh', isprint=False))
show_bgr(person_image, '人体检测')
# det_body_l 示例：wf(task='det_body_l').inference(str(MY_IMAGE), img_type='cv2', thr=0.25)

In [ ]:
# 常见物体检测：det_coco；把 task 改为 det_coco_l 可使用更大的模型。
coco = wf(task='det_coco')
coco_boxes, coco_image = coco.inference(str(require_file(MY_IMAGE, '图片')), img_type='cv2', thr=0.25)
print(coco.format_output(lang='zh', isprint=False))
show_bgr(coco_image, '物体检测')
# det_coco_l 示例：wf(task='det_coco_l').inference(str(MY_IMAGE), img_type='cv2', thr=0.25)

In [ ]:
# 手部检测：det_hand 使用 Palm ONNX，直接在整张图片上寻找多个手掌。
hand = wf(task='det_hand')
hand_boxes, hand_image = hand.inference(str(require_file(MY_IMAGE, '图片')), img_type='cv2', thr=0.6)
print('det_hand 检测框数量：', len(hand_boxes))
if len(hand_boxes) == 0:
    print('没有检测到手，请换一张手部更清晰的图片或降低 thr。')
face = wf(task='det_face')
face_boxes, face_image = face.inference(str(require_file(MY_IMAGE, '图片')), img_type='cv2', thr=0.4)
show_bgr(hand_image, '手部检测 det_hand')
show_bgr(face_image, '人脸检测 det_face')

In [ ]:
# 人体关键点：先运行人体检测单元，再把第一个检测框传入。
if len(person_boxes) == 0:
    raise ValueError('没有检测到人体。请换一张人物更清晰的图片，或降低 det_body 的阈值。')
person_box = np.asarray(person_boxes[0], dtype=np.float32)
for task in ['pose_body17', 'pose_body17_l', 'pose_body26', 'pose_wholebody133']:
    pose = wf(task=task)
    points, pose_image = pose.inference(str(MY_IMAGE), img_type='cv2', bbox=person_box)
    print(task, '关键点形状：', np.asarray(points).shape)
    show_bgr(pose_image, task)

In [ ]:
# 手部关键点：使用人体框和 wholebody133，一次得到人体、脸和左右手各 21 个点。
if len(person_boxes) == 0:
    raise ValueError('没有检测到人体，无法提取手部关键点。')
whole = wf(task='pose_wholebody133')
whole_points, whole_image = whole.inference(str(MY_IMAGE), img_type='cv2', bbox=person_box)
whole_points = np.asarray(whole_points)
left_hand21 = whole_points[91:112]
right_hand21 = whole_points[112:133]
print('左手关键点：', left_hand21.shape, '右手关键点：', right_hand21.shape)
show_bgr(whole_image, 'wholebody133（含左右手关键点）')

# 人脸关键点仍使用人脸检测框。
if 'face_boxes' not in globals():
    face = wf(task='det_face')
    face_boxes, face_image = face.inference(str(require_file(MY_IMAGE, '图片')), img_type='cv2', thr=0.4)

# 若使用近距离单手图片且 det_hand 有框，也可单独运行 pose_hand21：
if len(hand_boxes):
    hand_pose = wf(task='pose_hand21')
    hand_points, hand_pose_image = hand_pose.inference(str(MY_IMAGE), img_type='cv2', bbox=np.asarray(hand_boxes[0], dtype=np.float32))
    print('近景 pose_hand21:', np.asarray(hand_points).shape)
    show_bgr(hand_pose_image, '近景手部 21 点')
if len(face_boxes):
    landmark = wf(task='pose_face')
    face_points, face_landmark_image = landmark.inference(str(MY_IMAGE), img_type='cv2', bbox=np.asarray(face_boxes[0], dtype=np.float32))
    print('pose_face:', np.asarray(face_points).shape)
    show_bgr(face_landmark_image, '人脸关键点')
# pose_face106 兼容入口需要自己的 checkpoint：
# legacy = wf(task='pose_face106', checkpoint='/path/to/face106.onnx')

In [ ]:
# 图片分类：返回最可能的 ImageNet 类别。
classifier = wf(task='cls_imagenet')
scores, classified_image = classifier.inference(str(require_file(MY_IMAGE, '图片')), img_type='cv2')
print(classifier.format_output(lang='zh', isprint=False))
show_bgr(classified_image, '图像分类输入')

### OCR、分割、深度与道路感知

安装单元使用 `[all]`，已经包含 OCR 依赖。道路感知请替换为前方道路照片。

In [ ]:
# OCR：识别海报、试卷或手写清晰的文字图片。安装单元的 [all] 会带上 OCR 依赖。
ocr = wf(task='ocr')
ocr_result, ocr_image = ocr.inference(str(require_file(MY_OCR_IMAGE, 'OCR 图片')), img_type='cv2')
print(ocr.format_output(lang='zh', isprint=False))
show_bgr(ocr_image, 'OCR 结果')

In [ ]:
# 用人体框做分割；请先运行人体检测单元。
segmenter = wf(task='segment_anything')
masks, segmented_image = segmenter.inference(str(MY_IMAGE), mode='box', prompt=person_box, img_type='cv2')
print('掩码形状：', np.asarray(masks).shape)
show_bgr(segmented_image, '人物分割')

depth = wf(task='depth_anything')
depth_map, _ = depth.inference(str(MY_IMAGE), img_type='cv2')
plt.figure(figsize=(10, 6)); plt.imshow(np.squeeze(depth_map), cmap='inferno'); plt.colorbar(label='相对深度'); plt.axis('off'); plt.show()

In [ ]:
# 驾驶感知：车辆框、车道线和可行驶区域。
driving = wf(task='drive_perception')
driving_result, driving_image = driving.inference(str(require_file(MY_ROAD_IMAGE, '道路图片')), img_type='cv2', thr=0.25)
boxes, lane_mask, area_mask = driving_result
print('目标框、车道线、可行驶区域：', np.asarray(boxes).shape, np.asarray(lane_mask).shape, np.asarray(area_mask).shape)
show_bgr(driving_image, '道路感知')

## 5. 改变一张图片

风格迁移提供五种内置风格；图像着色适合黑白照片。

In [ ]:
# 内置风格：gen_style，以及 gen_style_mosaic、gen_style_candy、gen_style_rain-princess、gen_style_udnie、gen_style_pointilism。
for style in ['mosaic', 'candy', 'rain-princess', 'udnie', 'pointilism']:
    style_model = wf(task='gen_style', style=style)
    styled, _ = style_model.inference(str(MY_IMAGE), img_type='cv2')
    show_bgr(styled, f'风格迁移：{style}')
# 指定内部风格任务的写法：wf(task='gen_style_mosaic')
# gen_style_custom 需要自己的 checkpoint：wf(task='gen_style_custom', checkpoint='/path/to/style.onnx')

In [ ]:
# 图像着色：先准备灰度图片。
gray_path = OUTPUT_DIR / 'my-gray-image.png'
gray = cv2.imread(str(require_file(MY_IMAGE, '图片')), cv2.IMREAD_GRAYSCALE)
cv2.imwrite(str(gray_path), gray)
colorizer = wf(task='gen_color')
colored, _ = colorizer.inference(str(gray_path), img_type='cv2')
show_bgr(colored, '图像着色')

## 6. 比较图片、文字和声音

Embedding 会把内容转换成向量，便于比较相似度；原型分类则让你用自己的类别样例做分类。

### 先看懂输出

- `embedding_*: (数量, 维度)` 只表示成功把几份素材编码成向量，不表示它们相似或不相似。
- 相似度要在**同一种比较任务**里看排序：同一张图片、同一句文本或同一个音频与自己比较，通常是最高分；其余分数越高，表示模型认为它们越接近。
- 分数不是正确率，也不是百分比；不同模型、不同模态的分数不能横向比较。例如，图像相似度 `0.6` 不能与音频分类分数 `0.6` 判断为“一样相似”。
- 原型分类只会从你提供的标签中选择最接近的一类。它适合比较“更像哪一类”，不等于模型确认内容一定属于该类。

In [ ]:
# 图片 embedding 与相似度。
image_names = ['课堂图片', '道路图片']
image_embedder = wf(task='embedding_image')
image_vectors = image_embedder.inference([str(MY_IMAGE), str(MY_ROAD_IMAGE)])
image_similarity = get_similarity(image_vectors, image_vectors, method='cosine', use_softmax=False)
print('embedding_image:', image_vectors.shape)
print('行和列的顺序：', image_names)
print('图片相似度矩阵：\n', np.round(image_similarity, 3))

In [ ]:
# 文本 embedding 和用原型进行文本分类。
text_names = [MY_TEXT, '一辆汽车行驶在道路上']
text_embedder = wf(task='embedding_text')
text_vectors = text_embedder.inference(text_names)
text_similarity = get_similarity(text_vectors, text_vectors, method='cosine', use_softmax=False)
print('embedding_text:', text_vectors.shape)
print('文本相似度矩阵：\n', np.round(text_similarity, 3))
text_classifier = wf(task='cls_text')
text_result = text_classifier.inference(MY_TEXT, prototypes={'人工智能': ['图像识别', '机器学习'], '体育': ['篮球', '跑步']})
print('类别顺序：', text_result['labels'])
print('与各类别原型的相似度：', np.round(text_result['similarities'][0], 3))
print('最相近类别：', text_result['predictions'][0])

In [ ]:
# 图文匹配：候选文本中哪一句最符合图片。
candidate_texts = ['教师在教室里上课', '道路上的汽车', '一张白纸']
matcher = wf(task='match_image_text')
result = matcher.inference(str(MY_IMAGE), texts=candidate_texts)
print('候选文本及其分数：')
for text, score in zip(result['texts'], result['similarities'][0]):
    print(f'  {text}: {score:.3f}')
print('最佳匹配：', result['matches'][0]['text'], f"（{result['matches'][0]['score']:.3f}）")

### 音频功能

安装单元使用 `[all]`，已经包含音频依赖。音频示例会自动生成 WAV；需要时把 `MY_AUDIO` 换成自己的音频文件。

当前示例把 `MY_AUDIO` 同时作为待测音频和唯一的“演示音频”原型，因此它会被选为最相近类别，分数也通常接近 `1`。这只能验证流程可运行，**不能**证明模型已经会识别声音。课堂使用时，应为每个标签准备不同的参考录音（例如“掌声”“铃声”），再比较待测音频与这些参考录音的排序。

In [ ]:
# 音频 embedding、原型分类和关键词/事件检测。这里先用自动生成的示例音频；换成自己的 WAV 只需修改 MY_AUDIO。
audio_prototypes = {'演示音频': str(MY_AUDIO)}  # 可改为 {'掌声': 'clap.wav', '铃声': 'bell.wav'}
audio_embedder = wf(task='embedding_audio')
audio_vectors = audio_embedder.inference([str(require_file(MY_AUDIO, '音频文件'))])
print('embedding_audio:', np.asarray(audio_vectors).shape)
audio_classifier = wf(task='cls_audio')
audio_result = audio_classifier.inference(str(MY_AUDIO), prototypes=audio_prototypes)
print('类别顺序：', audio_result['labels'])
print('与各类别原型的相似度：', np.round(audio_result['similarities'][0], 3))
print('最相近类别：', audio_result['predictions'][0])
keyword_detector = wf(task='det_audio_keyword')
keyword_result = keyword_detector.inference(str(MY_AUDIO), prototypes=audio_prototypes, threshold=0.0, top_k=1)
print('达到阈值的声音事件：', keyword_result['detections'][0])

## 7. 从材料中找答案

`nlp_qa` 根据你给出的上下文提取答案，不会联网，也不会调用 LLM。

In [ ]:
context = 'XEdu-python 是面向中小学人工智能教学的工具包，提供视觉、文本、音频和多模态推理接口。'
question = 'XEdu-python 提供哪些推理接口？'
qa = wf(task='nlp_qa')
answer = qa.inference(question, context=context)
print(answer)

## 8. 高级扩展：使用你自己的模型或大语言模型

以下功能需要你提供自己的模型文件、模型仓库或 API Key。它们不是第一次使用时必须完成的内容。

In [ ]:
# MMEdu 导出的 ONNX：mmedu = wf(task='mmedu', checkpoint='/path/to/exported.onnx')
# BaseNN 导出的 ONNX：basenn = wf(task='basenn', checkpoint='/path/to/basenn.onnx')
# BaseML 导出的 PKL：baseml = wf(task='baseml', checkpoint='/path/to/model.pkl')
# 自定义 ONNX：custom = wf(task='custom', checkpoint='/path/to/custom.onnx')
# 模型仓库：repo_model = wf(repo='owner/repository', download_path='repo_cache')
# LLM（不要把 key 写入 Notebook）：
# import os
# from XEdu.LLM import Client
# client = Client(provider='qwen', api_key=os.environ['QWEN_API_KEY'])
# print(client.inference('请介绍 XEdu-python', stream=False))

## 下一步

把 `MY_IMAGE`、`MY_TEXT` 或 `MY_AUDIO` 换成自己的素材，选择一个功能重新运行，并将需要保留的图片保存到 `OUTPUT_DIR`。课堂中可以让学生比较：同一张图片在检测、关键点、分割和深度估计任务中的结果有什么不同？